# Nidhi's Contribution

Metrics Comparison for Tuning

In [51]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

def load_model_from_ckpt(ckpt_path):
    encoder = Encoder().to(device)
    decoder = Decoder().to(device)
    denoiser = UNet().to(device)

    checkpoint = torch.load(ckpt_path, map_location=device)
    encoder.load_state_dict(checkpoint["encoder"])
    decoder.load_state_dict(checkpoint["decoder"])
    denoiser.load_state_dict(checkpoint["denoiser"])

    encoder.eval()
    decoder.eval()
    denoiser.eval()
    return encoder, decoder, denoiser

def generate_images(decoder, n=64, latent_dim=(512, 8, 8)):
    generated = []
    for _ in range(n):
        z = torch.randn(1, *latent_dim).to(device)
        with torch.no_grad():
            img = decoder(z)
            img = (img.clamp(-1, 1) + 1) / 2  # normalize to [0,1]
            generated.append(img.squeeze(0))
    return torch.stack(generated)

def evaluate_model(generated_imgs, real_imgs):
    # IS
    is_metric = InceptionScore(normalize=True).to(device)
    is_score = is_metric(generated_imgs)

    # FID (convert to uint8)
    fid = FrechetInceptionDistance(feature=2048).to(device)
    real_uint8 = (real_imgs * 255).clamp(0, 255).to(torch.uint8)
    gen_uint8 = (generated_imgs * 255).clamp(0, 255).to(torch.uint8)
    fid.update(real_uint8, real=True)
    fid.update(gen_uint8, real=False)
    fid_score = fid.compute()

    return is_score, fid_score

# Step 1: Load real images (one batch)
real_imgs, _ = next(iter(dataloader))
real_imgs = real_imgs[:64].to(device)

# Step 2: Round 1
encoder1, decoder1, _ = load_model_from_ckpt("ldm_round1.pth")
gen_imgs1 = generate_images(decoder1, n=64)
is1, fid1 = evaluate_model(gen_imgs1, real_imgs)

# Step 3: Round 2
encoder2, decoder2, _ = load_model_from_ckpt("ldm_round2.pth")
gen_imgs2 = generate_images(decoder2, n=64)
is2, fid2 = evaluate_model(gen_imgs2, real_imgs)

# Step 4: Print results
print("📊 Comparison of LDM Rounds:")
print(f"Round 1 - Inception Score: {is1[0].item():.4f} ± {is1[1].item():.4f}, FID: {fid1.item():.4f}")
print(f"Round 2 - Inception Score: {is2[0].item():.4f} ± {is2[1].item():.4f}, FID: {fid2.item():.4f}")

📊 Comparison of LDM Rounds:
Round 1 - Inception Score: 1.1938 ± 0.0900, FID: 403.4338
Round 2 - Inception Score: 1.1214 ± 0.0515, FID: 414.0598


In [54]:
def load_model_from_ckpt(ckpt_path):
    encoder = Encoder().to(device)
    decoder = Decoder().to(device)
    denoiser = UNet().to(device)

    checkpoint = torch.load(ckpt_path, map_location=device)

    # Compatible with both key formats
    encoder_key = 'encoder' if 'encoder' in checkpoint else 'encoder_state_dict'
    decoder_key = 'decoder' if 'decoder' in checkpoint else 'decoder_state_dict'
    denoiser_key = 'denoiser' if 'denoiser' in checkpoint else 'denoiser_state_dict'

    encoder.load_state_dict(checkpoint[encoder_key])
    decoder.load_state_dict(checkpoint[decoder_key])
    denoiser.load_state_dict(checkpoint[denoiser_key])

    encoder.eval()
    decoder.eval()
    denoiser.eval()
    return encoder, decoder, denoiser

LDM Metrics: Model vs Tuning

In [57]:
# Load real images
real_imgs, _ = next(iter(dataloader))
real_imgs = real_imgs[:64].to(device)

# Model paths
checkpoints = {
    "LDM Model": "ldm_model.pth",
    "Round 1 Tuning": "ldm_round1.pth",
    "Round 2 Tuning": "ldm_round2.pth"
}

results = {}

# Evaluate all three models
for label, path in checkpoints.items():
    encoder, decoder, _ = load_model_from_ckpt(path)
    gen_imgs = generate_images(decoder, n=64)
    is_score, fid_score = evaluate_model(gen_imgs, real_imgs)

    results[label] = {
        "IS": is_score[0].item(),
        "IS_std": is_score[1].item(),
        "FID": fid_score.item()
    }

# Print Results
print("📊 Comparison of Inception Score and FID")
print("{:<10} | {:>6} ± {:<5} | {:>7}".format("Model", "IS", "Std", "FID"))
print("-" * 40)
for label, metrics in results.items():
    print(f"{label:<10} | {metrics['IS']:>6.3f} ± {metrics['IS_std']:<5.3f} | {metrics['FID']:>7.2f}")

📊 Comparison of Inception Score and FID
Model      |     IS ± Std   |     FID
----------------------------------------
LDM Model  |  1.168 ± 0.090 |  391.50
Round 1 Tuning |  1.173 ± 0.083 |  398.08
Round 2 Tuning |  1.126 ± 0.059 |  404.68


LLM Prompts:
*   How to find Inception Score and Frechet Inception Distance for evaluating generative models?
*   How to compare metrics between multiple saved model checkpoints? (original model, tuning round 1 and 2)